# Prepare training datasets (3 models)

Single notebook: from the **definitive** zone CSV, export training packs for:

| Model | Output |
|-------|--------|
| **Informer2020** | Wide CSV per station |
| **AirFormer** | `train/val/test.npz` + adjacency pickle |
| **GAT-Informer** | `data.npz` (windows + graph) |

Shared choices: `ZONE`, `CONTAMINANT`, F-levels, `seq_len`, `horizon`, temporal split.

1. Set config (or load recommendation from feature selection).
2. Run all cells.
3. Check the manifest at the end.


In [1]:
# ===== CONFIG =====
ZONE = 2
CONTAMINANT = "PM10"
START_DATE = "2019-01-01"
MIN_AVAILABILITY = 80.0

# If None, load recommended ablation from feature_selection_summary.json
FEATURE_CONFIGS = None  # e.g. ["F1", "F3", "F5"]

SEQ_LEN = None          # if None, load from ACF / feature-selection JSON
HORIZON = 24
LABEL_LEN = None        # Informer2020 decoder label length; default SEQ_LEN // 2

TRAIN_RATIO = 0.70
VAL_RATIO = 0.10
# test = remainder

GRAPH_CSV = None  # default: data/graphs/meteo_graph_zone_{ZONE}_top5.csv
INTERPOLATE_LIMIT = 3

EXPORT_INFORMER2020 = True
EXPORT_AIRFORMER = True
EXPORT_GAT_INFORMER = True

from pathlib import Path
import json
import pickle
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "data" / "training").is_dir():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not find data/training/. Open from MASTER repo.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.feature_configs import (
    ABLATION_LEVELS,
    get_feature_config,
    resolve_features_in_dataset,
)

DATASET_PATH = ROOT / f"data/training/zone_{ZONE}/dataset_zone_{ZONE}_{CONTAMINANT}.csv"
FS_SUMMARY = ROOT / f"analysis/feature_selection/zone_{ZONE}/{CONTAMINANT}/feature_selection_summary.json"
ACF_JSON = ROOT / f"analysis/acf/zone_{ZONE}/{CONTAMINANT}/seq_len_recommendation.json"
OUT_ROOT = ROOT / f"data/training/zone_{ZONE}"
GRAPH_CSV = Path(GRAPH_CSV) if GRAPH_CSV else ROOT / f"data/graphs/meteo_graph_zone_{ZONE}_top5.csv"
STATION_MAP_CSV = ROOT / "data/metadata/stations_meteo_graph_check.csv"

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATASET_PATH}")

# Load recommendations
if FEATURE_CONFIGS is None:
    if FS_SUMMARY.exists():
        fs = json.loads(FS_SUMMARY.read_text(encoding="utf-8"))
        FEATURE_CONFIGS = fs.get("feature_config_recommendation", {}).get(
            "ablation_to_train", ["F1", "F3", "F5"]
        )
        print(f"FEATURE_CONFIGS from feature selection: {FEATURE_CONFIGS}")
    else:
        FEATURE_CONFIGS = ["F1", "F3", "F5"]
        print(f"FEATURE_CONFIGS default: {FEATURE_CONFIGS}")

for fc in FEATURE_CONFIGS:
    if fc not in ABLATION_LEVELS:
        raise ValueError(f"Unknown feature config {fc}")

if SEQ_LEN is None:
    if ACF_JSON.exists():
        SEQ_LEN = int(json.loads(ACF_JSON.read_text(encoding="utf-8"))["recommended_seq_len"])
        print(f"SEQ_LEN from ACF JSON: {SEQ_LEN}")
    elif FS_SUMMARY.exists():
        SEQ_LEN = int(json.loads(FS_SUMMARY.read_text(encoding="utf-8"))["seq_len"])
        print(f"SEQ_LEN from feature selection: {SEQ_LEN}")
    else:
        SEQ_LEN = 48
        print(f"SEQ_LEN default: {SEQ_LEN}")

if LABEL_LEN is None:
    LABEL_LEN = SEQ_LEN // 2

print(f"Zone={ZONE} Contaminant={CONTAMINANT}")
print(f"Dataset={DATASET_PATH.relative_to(ROOT)}")
print(f"seq_len={SEQ_LEN} horizon={HORIZON} label_len={LABEL_LEN}")
print(f"Graph={GRAPH_CSV.relative_to(ROOT) if GRAPH_CSV.exists() else 'MISSING'}")


FEATURE_CONFIGS from feature selection: ['F1', 'F3', 'F5']
SEQ_LEN from ACF JSON: 48
Zone=2 Contaminant=PM10
Dataset=data\training\zone_2\dataset_zone_2_PM10.csv
seq_len=48 horizon=24 label_len=24
Graph=data\graphs\meteo_graph_zone_2_top5.csv


## 1. Shared helpers


In [2]:
def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["time"] = pd.to_datetime(out["time"])
    out["hour"] = out["time"].dt.hour
    out["dow"] = out["time"].dt.dayofweek
    out["month"] = out["time"].dt.month
    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)
    out["dow_sin"] = np.sin(2 * np.pi * out["dow"] / 7)
    out["dow_cos"] = np.cos(2 * np.pi * out["dow"] / 7)
    out["month_sin"] = np.sin(2 * np.pi * out["month"] / 12)
    out["month_cos"] = np.cos(2 * np.pi * out["month"] / 12)
    if "DV" in out.columns:
        out["wind_sin"] = np.sin(np.deg2rad(out["DV"]))
        out["wind_cos"] = np.cos(np.deg2rad(out["DV"]))
    return out


def interpolate_numeric(df: pd.DataFrame, limit: int = 3) -> pd.DataFrame:
    out = df.copy()
    skip = {"time", "ID", "station_name", "lat", "lon", "zone", "hour", "dow", "month"}
    cols = [c for c in out.columns if c not in skip and pd.api.types.is_numeric_dtype(out[c])]
    out[cols] = out.groupby("ID")[cols].transform(
        lambda s: s.interpolate(limit=limit)
    )
    return out


def station_key(name: str) -> str:
    return name.replace(" ", "_").replace("(", "").replace(")", "")


def resolve_cols(df: pd.DataFrame, level: str):
    used, excluded, avail = resolve_features_in_dataset(
        df, CONTAMINANT, level, min_availability_pct=MIN_AVAILABILITY
    )
    return used, excluded, avail


def temporal_split_indices(n_samples: int, train_ratio: float, val_ratio: float):
    n_train = int(n_samples * train_ratio)
    n_val = int(n_samples * val_ratio)
    n_test = n_samples - n_train - n_val
    if min(n_train, n_val, n_test) <= 0:
        raise ValueError(f"Split too small for n_samples={n_samples}")
    return n_train, n_val, n_test


print("Helpers ready.")


Helpers ready.


## 2. Load definitive dataset


In [3]:
raw = pd.read_csv(DATASET_PATH, parse_dates=["time"], low_memory=False)
raw = raw[raw["time"] >= pd.Timestamp(START_DATE)].copy()
df = add_derived_features(raw)
df = interpolate_numeric(df, limit=INTERPOLATE_LIMIT)
df = df.sort_values(["time", "ID"]).reset_index(drop=True)

stations = sorted(df["station_name"].dropna().unique().tolist())
print(f"Rows: {len(df):,} | stations: {len(stations)} | "
      f"{df['time'].min()} → {df['time'].max()}")
print(stations)


Rows: 994,080 | stations: 18 | 2019-08-30 00:00:00 → 2025-12-29 23:00:00
['ABANTO', 'ALGORTA (BBIZI2)', 'ALONSOTEGI', 'ARRAIZ (Monte)', 'BARAKALDO', 'BASAURI', 'CASTREJANA', 'ERANDIO', 'EUROPA', 'LAS CARRERAS', 'MAZARREDO', 'MUSKIZ', 'Mª DIAZ HARO', 'SAN JULIAN', 'SAN MIGUEL', 'SANGRONIZ', 'SANTURCE', 'ZIERBENA (Puerto)']


## 3. Informer2020 — wide CSV per station

Layout: `data/training/zone_<n>/informer2020/<STATION>/<F*>/data.csv`  
Columns: `date, feat_1, ..., target` (target last).


In [4]:
informer_manifest = []

if EXPORT_INFORMER2020:
    for level in FEATURE_CONFIGS:
        # Resolve features PER STATION (zone-wide lists drop stations that lack
        # co-pollutants, e.g. SAN MIGUEL / F5 with NO2/NOx).
        target = CONTAMINANT
        for station in stations:
            sdf = df[df["station_name"] == station].copy().sort_values("time")
            feature_cols, excluded, _ = resolve_cols(sdf, level)
            if target not in feature_cols:
                print(f"  skip {station} / {level}: target missing after resolve")
                continue
            input_cols = [c for c in feature_cols if c != target]
            export_cols = input_cols + [target]
            wide = sdf[["time"] + export_cols].rename(columns={"time": "date"})
            wide = wide.dropna(subset=export_cols)
            if len(wide) < SEQ_LEN + HORIZON:
                print(f"  skip {station} / {level}: too few rows ({len(wide)})")
                continue

            out_dir = OUT_ROOT / "informer2020" / station_key(station) / level
            out_dir.mkdir(parents=True, exist_ok=True)
            csv_path = out_dir / "data.csv"
            wide.to_csv(csv_path, index=False)

            meta = {
                "model": "informer2020",
                "station": station,
                "zone": ZONE,
                "contaminant": CONTAMINANT,
                "feature_config": level,
                "features": export_cols,
                "features_excluded": excluded,
                "n_rows": int(len(wide)),
                "seq_len": SEQ_LEN,
                "label_len": LABEL_LEN,
                "pred_len": HORIZON,
                "path": str(csv_path.relative_to(ROOT)).replace("\\", "/"),
            }
            (out_dir / "metadata.json").write_text(
                json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8"
            )
            informer_manifest.append(meta)

    print(f"Informer2020 exports: {len(informer_manifest)} station×config files")
else:
    print("Informer2020 export skipped.")


  skip SAN MIGUEL / F5: too few rows (0)
Informer2020 exports: 53 station×config files


## 4. Build station adjacency (AirFormer + GAT-Informer)

Uses `meteo_graph_zone_*_top5.csv` mapped through `stations_meteo_graph_check.csv`.


In [5]:
def build_adjacency(stations_ordered, graph_csv: Path, map_csv: Path):
    if not graph_csv.exists():
        raise FileNotFoundError(graph_csv)
    if not map_csv.exists():
        raise FileNotFoundError(map_csv)

    id_map = pd.read_csv(map_csv)
    # id like S1 → station name
    sid_to_name = dict(zip(id_map["id"].astype(str), id_map["station"].astype(str)))
    name_to_idx = {n: i for i, n in enumerate(stations_ordered)}

    edges = pd.read_csv(graph_csv)
    n = len(stations_ordered)
    adj = np.zeros((n, n), dtype=np.float64)
    used_edges = 0
    for _, row in edges.iterrows():
        s_name = sid_to_name.get(str(row["source"]))
        t_name = sid_to_name.get(str(row["target"]))
        if s_name not in name_to_idx or t_name not in name_to_idx:
            continue
        i, j = name_to_idx[s_name], name_to_idx[t_name]
        w = float(row["weight"])
        adj[i, j] = max(adj[i, j], w)
        adj[j, i] = max(adj[j, i], w)
        used_edges += 1

    # self-loops
    np.fill_diagonal(adj, 1.0)
    return adj, used_edges


# Prefer stations that appear both in data and graph map
map_df = pd.read_csv(STATION_MAP_CSV)
mapped_names = set(map_df["station"].astype(str))
spatial_stations = [s for s in stations if s in mapped_names]
if len(spatial_stations) < 2:
    spatial_stations = stations  # fallback
    print("WARNING: few stations matched graph map; using all dataset stations.")

adj, n_edges = build_adjacency(spatial_stations, GRAPH_CSV, STATION_MAP_CSV)
print(f"Spatial stations: {len(spatial_stations)} | edges used: {n_edges}")
print(f"Adj shape: {adj.shape} | density: {(adj > 0).mean():.3f}")


Spatial stations: 18 | edges used: 36
Adj shape: (18, 18) | density: 0.247


## 5. AirFormer — multivariate spatio-temporal windows

Format (paper-compatible):

- `x`: `(samples, seq_len, num_nodes, input_dim)`
- `y`: `(samples, horizon, num_nodes, 1)`  ← target only

Saved under `data/training/zone_<n>/airformer/<DATASET_NAME>/<F*>/`.


In [6]:
airformer_manifest = []


def pivot_feature_cube(df_in, stations_ordered, feature_cols):
    """Return array (T, N, F) aligned on common timestamps."""
    pieces = []
    for col in feature_cols:
        wide = (
            df_in.pivot_table(index="time", columns="station_name", values=col, aggfunc="mean")
            .reindex(columns=stations_ordered)
            .sort_index()
        )
        pieces.append(wide)
    # align on intersection of times with any data; keep rows where target not all-nan
    times = pieces[0].index
    for p in pieces[1:]:
        times = times.intersection(p.index)
    mats = [p.reindex(index=times).to_numpy(dtype=np.float32) for p in pieces]
    cube = np.stack(mats, axis=-1)  # (T, N, F)
    return times.to_numpy(), cube


def fill_cube_nans(cube, target_idx=0):
    """Time-wise ffill/bfill per node/feature; remaining exogenous NaN -> 0."""
    out = cube.copy()
    _t, n, f = out.shape
    for j in range(n):
        for k in range(f):
            s = pd.Series(out[:, j, k])
            s = s.ffill().bfill()
            out[:, j, k] = s.to_numpy(dtype=np.float32)
            if k != target_idx:
                out[:, j, k] = np.nan_to_num(out[:, j, k], nan=0.0)
    return out


def make_st_windows(cube, seq_len, horizon, target_idx):
    """cube (T,N,F) -> X (S,seq,N,F), Y (S,horizon,N,1)"""
    cube = fill_cube_nans(cube, target_idx=target_idx)
    t, n, f = cube.shape
    xs, ys = [], []
    max_start = t - seq_len - horizon + 1
    for i in range(max_start):
        x = cube[i : i + seq_len]
        y = cube[i + seq_len : i + seq_len + horizon, :, target_idx : target_idx + 1]
        if np.isnan(x[:, :, target_idx]).any() or np.isnan(y).any():
            continue
        x = np.nan_to_num(x, nan=0.0)
        xs.append(x)
        ys.append(y)
    if not xs:
        return np.empty((0, seq_len, n, f)), np.empty((0, horizon, n, 1))
    return np.stack(xs), np.stack(ys)


if EXPORT_AIRFORMER:
    for level in FEATURE_CONFIGS:
        feature_cols, excluded, _ = resolve_cols(df, level)
        # AirFormer input_dim = all resolved features; y = target channel
        if CONTAMINANT not in feature_cols:
            raise ValueError(f"AirFormer {level}: missing target")
        # order: put target first (matches AIR_TINY scaling of channel 0)
        ordered = [CONTAMINANT] + [c for c in feature_cols if c != CONTAMINANT]
        times, cube = pivot_feature_cube(df, spatial_stations, ordered)
        print(f"AirFormer {level}: cube {cube.shape} (T,N,F)")

        X, Y = make_st_windows(cube, SEQ_LEN, HORIZON, target_idx=0)
        print(f"  windows: X={X.shape} Y={Y.shape}")
        if len(X) == 0:
            print(f"  SKIP {level}: no complete windows")
            continue

        n_train, n_val, n_test = temporal_split_indices(len(X), TRAIN_RATIO, VAL_RATIO)
        splits = {
            "train": (X[:n_train], Y[:n_train]),
            "val": (X[n_train:n_train + n_val], Y[n_train:n_train + n_val]),
            "test": (X[n_train + n_val:], Y[n_train + n_val:]),
        }

        ds_name = f"ZONE{ZONE}_{CONTAMINANT}"
        out_dir = OUT_ROOT / "airformer" / ds_name / level
        out_dir.mkdir(parents=True, exist_ok=True)

        for cat, (x_cat, y_cat) in splits.items():
            np.savez_compressed(out_dir / f"{cat}.npz", x=x_cat, y=y_cat)

        sensor_ids = [station_key(s) for s in spatial_stations]
        sensor_id_to_ind = {sid: i for i, sid in enumerate(sensor_ids)}
        pkl_path = out_dir / "adj_mx.pkl"
        with open(pkl_path, "wb") as f:
            pickle.dump((sensor_ids, sensor_id_to_ind, adj.astype(np.float32)), f, protocol=4)

        meta = {
            "model": "airformer",
            "dataset_name": ds_name,
            "zone": ZONE,
            "contaminant": CONTAMINANT,
            "feature_config": level,
            "features": ordered,
            "features_excluded": excluded,
            "stations": spatial_stations,
            "num_nodes": len(spatial_stations),
            "seq_len": SEQ_LEN,
            "horizon": HORIZON,
            "n_train": int(n_train),
            "n_val": int(n_val),
            "n_test": int(n_test),
            "x_shape": list(X.shape),
            "y_shape": list(Y.shape),
            "path": str(out_dir.relative_to(ROOT)).replace("\\", "/"),
            "adj_path": str(pkl_path.relative_to(ROOT)).replace("\\", "/"),
            "note": "Register dataset in AirFormer get_num_nodes before training.",
        }
        (out_dir / "metadata.json").write_text(
            json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        airformer_manifest.append(meta)

    print(f"AirFormer exports: {len(airformer_manifest)}")
else:
    print("AirFormer export skipped.")


AirFormer F1: cube (55512, 18, 1) (T,N,F)
  windows: X=(55441, 48, 18, 1) Y=(55441, 24, 18, 1)
AirFormer F3: cube (55512, 18, 13) (T,N,F)
  windows: X=(55441, 48, 18, 13) Y=(55441, 24, 18, 1)
AirFormer F5: cube (55512, 18, 16) (T,N,F)
  windows: X=(55441, 48, 18, 16) Y=(55441, 24, 18, 1)
AirFormer exports: 3


## 6. GAT-Informer — target windows + graph

Format aligned with upstream `data.npz`:

- `train_x_raw` / `train_y`: `(samples, num_nodes, seq_len/horizon)`
- `vail_*`, `test_*`
- `graph`: `(N, N)`
- `max_min`: `[max, min]` of raw target

Uses the **target** series only (spatial model + Informer temporal), same F-level stations filter as availability-ready set.


In [7]:
gat_manifest = []


def make_node_windows(series_tn, seq_len, horizon):
    """series (T,N) -> X (S,N,seq), Y (S,N,horizon)"""
    t, n = series_tn.shape
    xs, ys = [], []
    for i in range(t - seq_len - horizon + 1):
        x = series_tn[i : i + seq_len].T  # (N, seq)
        y = series_tn[i + seq_len : i + seq_len + horizon].T
        if np.isnan(x).any() or np.isnan(y).any():
            continue
        xs.append(x)
        ys.append(y)
    if not xs:
        return np.empty((0, n, seq_len)), np.empty((0, n, horizon))
    return np.stack(xs), np.stack(ys)


if EXPORT_GAT_INFORMER:
    for level in FEATURE_CONFIGS:
        # Keep same station set as AirFormer; features_used only gates documentation
        feature_cols, excluded, _ = resolve_cols(df, level)
        times, cube = pivot_feature_cube(df, spatial_stations, [CONTAMINANT])
        series = cube[:, :, 0]  # (T, N)
        print(f"GAT-Informer {level}: series {series.shape}")

        # normalize with train-period max/min after windowing split
        X, Y = make_node_windows(series, SEQ_LEN, HORIZON)
        print(f"  windows: X={X.shape} Y={Y.shape}")
        if len(X) == 0:
            print(f"  SKIP {level}: no complete windows")
            continue

        n_train, n_val, n_test = temporal_split_indices(len(X), TRAIN_RATIO, VAL_RATIO)
        # max_min from training windows only (raw scale before norm)
        train_x_raw = X[:n_train]
        train_y = Y[:n_train]
        vmax = float(np.nanmax(train_x_raw))
        vmin = float(np.nanmin(train_x_raw))
        if vmax == vmin:
            vmax = vmin + 1.0

        def norm(a):
            return (a - vmin) / (vmax - vmin)

        out = {
            "train_x_raw": norm(X[:n_train]).astype(np.float32),
            "train_y": norm(Y[:n_train]).astype(np.float32),
            "vail_x_raw": norm(X[n_train:n_train + n_val]).astype(np.float32),
            "vail_y": norm(Y[n_train:n_train + n_val]).astype(np.float32),
            "test_x_raw": norm(X[n_train + n_val:]).astype(np.float32),
            "test_y": norm(Y[n_train + n_val:]).astype(np.float32),
            "max_min": np.array([vmax, vmin], dtype=np.float32),
            "graph": adj.astype(np.float32),
        }

        out_dir = OUT_ROOT / "gat_informer" / level
        out_dir.mkdir(parents=True, exist_ok=True)
        npz_path = out_dir / f"data{SEQ_LEN}.npz"
        np.savez_compressed(npz_path, **out)

        meta = {
            "model": "gat_informer",
            "zone": ZONE,
            "contaminant": CONTAMINANT,
            "feature_config": level,
            "note_features": (
                "Windows built from target only; F-level recorded for experiment parity. "
                f"Resolved extras (not in GAT tensors): "
                f"{[c for c in feature_cols if c != CONTAMINANT]}"
            ),
            "features_excluded": excluded,
            "stations": spatial_stations,
            "num_nodes": len(spatial_stations),
            "seq_len": SEQ_LEN,
            "horizon": HORIZON,
            "n_train": int(n_train),
            "n_val": int(n_val),
            "n_test": int(n_test),
            "x_shape": list(out["train_x_raw"].shape),
            "path": str(npz_path.relative_to(ROOT)).replace("\\", "/"),
        }
        (out_dir / "metadata.json").write_text(
            json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        gat_manifest.append(meta)

    print(f"GAT-Informer exports: {len(gat_manifest)}")
else:
    print("GAT-Informer export skipped.")


GAT-Informer F1: series (55512, 18)
  windows: X=(45501, 18, 48) Y=(45501, 18, 24)
GAT-Informer F3: series (55512, 18)
  windows: X=(45501, 18, 48) Y=(45501, 18, 24)
GAT-Informer F5: series (55512, 18)
  windows: X=(45501, 18, 48) Y=(45501, 18, 24)
GAT-Informer exports: 3


## 7. Manifest summary


In [8]:
manifest = {
    "zone": ZONE,
    "contaminant": CONTAMINANT,
    "dataset": str(DATASET_PATH.as_posix()),
    "start_date": START_DATE,
    "feature_configs": FEATURE_CONFIGS,
    "seq_len": SEQ_LEN,
    "horizon": HORIZON,
    "label_len": LABEL_LEN,
    "train_ratio": TRAIN_RATIO,
    "val_ratio": VAL_RATIO,
    "spatial_stations": spatial_stations,
    "graph_csv": str(GRAPH_CSV.as_posix()),
    "informer2020": informer_manifest,
    "airformer": airformer_manifest,
    "gat_informer": gat_manifest,
}

manifest_path = OUT_ROOT / f"prepare_training_manifest_{CONTAMINANT}.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

print("=" * 60)
print(f"ZONE {ZONE} · {CONTAMINANT} · seq_len={SEQ_LEN} · horizon={HORIZON}")
print(f"Configs: {FEATURE_CONFIGS}")
print("=" * 60)
print(f"Informer2020 : {len(informer_manifest)} files")
print(f"AirFormer    : {len(airformer_manifest)} configs")
print(f"GAT-Informer : {len(gat_manifest)} configs")
print(f"\nManifest: {manifest_path.relative_to(ROOT)}")
print("=" * 60)
print("Next: train Informer2020 on one station (e.g. BASAURI / F5), then compare.")


ZONE 2 · PM10 · seq_len=48 · horizon=24
Configs: ['F1', 'F3', 'F5']
Informer2020 : 53 files
AirFormer    : 3 configs
GAT-Informer : 3 configs

Manifest: data\training\zone_2\prepare_training_manifest_PM10.json
Next: train Informer2020 on one station (e.g. BASAURI / F5), then compare.
